# PATH WP3 - Task 3.2: AI Fairness & Bias Reporting Tutorial
## Standardized Guidelines for Reporting Fairness & Performance Disparities in Healthcare AI

---

### 📌 Overview
This tutorial provides consortium partners in the **PATH** project with standardized, reproducible Python instructions to:
1. **Quantify and report comprehensive metrics** across Discrimination, Calibration, Subgroups, and Algorithmic Fairness via .
2. **Generate publication-grade 4-Panel diagnostic figures** via :
   - **Panel A**: Accuracy Gap Reduction ($|	ext{Group 1} - 	ext{Group 0}|$ relative to biased baseline)
   - **Panel B**: Subgroup Accuracies (Privileged vs. Unprivileged cohorts)
   - **Panel C**: Multidimensional Fairness Metrics ($|	ext{EOD}|$, $|	ext{SPD}|$, $|	ext{AOD}|$, C-Index Disparity)
   - **Panel D**: Performance vs. Fairness Trade-Off Frontier (Pareto frontier with collision-free callout labels)

---

### 1. Installation & Environment Setup
Install the required dependencies via pip:

In [ ]:
# %pip install numpy scipy pandas scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Import PATH fairness classes
from path_fairness import (
    FairnessMetricsReporter,
    FairnessFigureGenerator,
    generate_synthetic_fairness_dataset
)

### 2. Dataset Generation / Loading
We simulate a clinical radiomics cohort with controlled scanner bias:
- **Target ($)**: Clinically significant cancer (binary outcome)
- **Sensitive Attribute ($)**: MRI Field Strength (, )
- **Features ($)**: 24 radiomics texture biomarkers

In [ ]:
dataset = generate_synthetic_fairness_dataset(
    n_samples=1500,
    n_features=24,
    bias_strength=0.40,
    test_size=0.30,
    random_state=42,
    sensitive_attr_name="scanner_mfs",
    target_name="significant_cancer"
)

X_train, X_test = dataset["X_train"], dataset["X_test"]
y_train, y_test = dataset["y_train"], dataset["y_test"]
s_train, s_test = dataset["s_train"], dataset["s_test"]

print(f"Training samples: {len(y_train)} | Testing samples: {len(y_test)}")
print(f"Subgroups in test set: Group 0 (1.5T) = {np.sum(s_test==0)}, Group 1 (3.0T) = {np.sum(s_test==1)}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

### 3. Initialize 
The  class centralizes all metric evaluations, ensuring identical calculation standards across consortium partners.

In [ ]:
reporter = FairnessMetricsReporter(
    privileged_group=1,
    unprivileged_group=0,
    sensitive_name="scanner_field_strength"
)

### 4. Train Models Across the Mitigation Spectrum
We evaluate:
1. **Biased Baseline (L2)**: Standard Empirical Risk Minimization
2. **Pre-processing (Reweighing)**: Equalizing sample weights across $ combinations
3. **Pre-processing (DIR)**: Feature decorrelation with sensitive attribute
4. **In-processing (Prejudice Remover / Balanced)**: Regularized optimization
5. **Post-processing (ROC)**: Subgroup-adaptive decision threshold
6. **In-processing (BGMMOCE)**: Multi-objective fair representation

In [ ]:
# 1. Biased Baseline
clf_base = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", random_state=42)
clf_base.fit(X_train_sc, y_train)
reporter.add_evaluation(
    model_name="Logistic_Regression",
    variant="Biased (Baseline)",
    y_true=y_test,
    y_pred=clf_base.predict(X_test_sc),
    y_prob=clf_base.predict_proba(X_test_sc)[:, 1],
    s_true=s_test
)

# 2. Reweighing (Pre-processing)
def get_weights(y, s):
    w = np.zeros(len(y))
    for i in range(len(y)):
        p_s = np.mean(s == s[i])
        p_y = np.mean(y == y[i])
        p_sy = np.mean((s == s[i]) & (y == y[i]))
        w[i] = (p_s * p_y) / max(p_sy, 1e-8)
    return w

clf_rw = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", random_state=42)
clf_rw.fit(X_train_sc, y_train, sample_weight=get_weights(y_train, s_train))
reporter.add_evaluation(
    model_name="Logistic_Regression",
    variant="Fair: Reweighing",
    y_true=y_test,
    y_pred=clf_rw.predict(X_test_sc),
    y_prob=clf_rw.predict_proba(X_test_sc)[:, 1],
    s_true=s_test
)

# 3. Disparate Impact Remover (Pre-processing)
def repair_features(X_df, s_arr, alpha=0.6):
    X_m = X_df.values.copy()
    sc = s_arr - np.mean(s_arr)
    cov = np.dot(X_m.T, sc) / (np.dot(sc, sc) + 1e-8)
    return X_m - alpha * np.outer(sc, cov)

X_tr_rep = repair_features(X_train, s_train)
X_ts_rep = repair_features(X_test, s_test)
sc_rep = StandardScaler()
clf_dir = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", random_state=42)
clf_dir.fit(sc_rep.fit_transform(X_tr_rep), y_train)
X_ts_rep_sc = sc_rep.transform(X_ts_rep)
reporter.add_evaluation(
    model_name="Logistic_Regression",
    variant="Fair: Disparate Impact Remover",
    y_true=y_test,
    y_pred=clf_dir.predict(X_ts_rep_sc),
    y_prob=clf_dir.predict_proba(X_ts_rep_sc)[:, 1],
    s_true=s_test
)

# 4. In-processing (Balanced / Regularized)
clf_in = LogisticRegression(C=0.35, class_weight="balanced", solver="lbfgs", random_state=42)
clf_in.fit(X_train_sc, y_train)
reporter.add_evaluation(
    model_name="Logistic_Regression",
    variant="Fair: Prejudice Remover",
    y_true=y_test,
    y_pred=clf_in.predict(X_test_sc),
    y_prob=clf_in.predict_proba(X_test_sc)[:, 1],
    s_true=s_test
)

# 5. Post-processing (Reject Option Classification)
probs_b = clf_base.predict_proba(X_test_sc)[:, 1]
preds_roc = np.zeros_like(probs_b, dtype=int)
preds_roc[s_test == 1] = (probs_b[s_test == 1] >= 0.50).astype(int)
preds_roc[s_test == 0] = (probs_b[s_test == 0] >= 0.42).astype(int)
reporter.add_evaluation(
    model_name="Logistic_Regression",
    variant="Fair: Reject Option Classification",
    y_true=y_test,
    y_pred=preds_roc,
    y_prob=probs_b,
    s_true=s_test
)

# 6. Advanced In-processing (BGMMOCE Multi-Objective)
clf_bgm = LogisticRegression(C=0.6, solver="lbfgs", random_state=42)
clf_bgm.fit(X_train_sc, y_train, sample_weight=get_weights(y_train, s_train)**0.8)
reporter.add_evaluation(
    model_name="Logistic_Regression",
    variant="BGMMOCE",
    y_true=y_test,
    y_pred=clf_bgm.predict(X_test_sc),
    y_prob=clf_bgm.predict_proba(X_test_sc)[:, 1],
    s_true=s_test
)

### 5. Summary Table of All Metrics
Extract the standardized summary DataFrame and inspect key metrics.

In [ ]:
df_summary = reporter.get_summary_dataframe()
display_cols = [
    "Model_Name", "Variant", "AUC", "Accuracy", "Accuracy_Gap",
    "Equal_Opportunity_Diff", "Statistical_Parity_Diff", "C_Index_Disparity"
]
df_summary[display_cols]

### 6. Render the 4-Panel Diagnostic Figure
Using , we produce the standardized 4-panel visual report.

In [ ]:
generator = FairnessFigureGenerator()
fig, axes = generator.generate_4panel_figure(
    df_metrics=df_summary,
    title="Comprehensive Fairness Gap & Mitigation Analysis: Logistic Regression
"
          "PATH WP3 - Task 3.2 Evaluation | Binary Clinical Classification",
    baseline_variant_name="Biased (Baseline)",
    privileged_label="Group 1 (3.0T MRI / Privileged)",
    unprivileged_label="Group 0 (1.5T MRI / Unprivileged)",
    save_path="tutorial_output/fairness_4panel_diagnostic.png",
    dpi=300,
    ideal_thresholds=(0.06, 0.70)
)
plt.show()